In [40]:
from data.cleaning import read_csv
from core import Config
import pandas as pd

config = Config()

static_df: pd.DataFrame = read_csv(
    config.eda_filtered_dir / "eda_filtered_static_combined_2026.csv",
    config.eda_filtered_dir / "eda_filtered_static_dtypes_2026.csv",
    [0]
)

historic_df: pd.DataFrame = read_csv(
    config.eda_filtered_dir / "eda_filtered_historic_2026.csv",
    config.eda_filtered_dir / "eda_filtered_historic_dtypes_2026.csv",
    [0]
)

# Impute static dataframe

In [41]:
from data.constants.constants_hq import HQ

'''
Example how one loop of fill_na_by_modes looks like:

modes: dict[str, str] = {}
for group in filtered_df.groupby("TR.HQCountryCode")['Currency Code']:
    country = group[0]
    currency = group[1].mode().iloc[0]
    modes[country] = currency
mask = filtered_df['Currency Code'].isna()
filtered_df.loc[mask, 'Currency Code'] = filtered_df.loc[mask, 'TR.HQCountryCode'].map(modes)
'''

def fill_na_by_modes(df: pd.DataFrame) -> pd.DataFrame:
    fill_key_by_modes_of_value: dict[str, str] = {
        'Currency Code': 'TR.HQCountryCode',
        'TR.AssetCategory': 'TR.GICSSectorCode',
        'TR.BusinessSector': 'TR.GICSSectorCode',
        'TR.BusinessSectorScheme': 'TR.GICSSectorCode',
        'TR.CompanyParentType': 'TR.GICSSectorCode',
        'TR.HeadquartersRegionAlt': 'TR.HQCountryCode',
        'TR.InstrumentType': 'TR.GICSSectorCode',
        'TR.OrganizationType': 'TR.GICSSectorCode',
        'TR.PriceMainIndex': 'TR.HQCountryCode',
        'TR.RelatedOrgISO2': 'TR.HQCountryCode',
        'TR.RelatedOrgType': 'TR.GICSSectorCode',
    }
    df = df.copy()
    modes: dict[str, str]
    mask: pd.Series

    for missing_value_col, col in fill_key_by_modes_of_value.items():
        modes = {}

        groups = df.groupby(col)[missing_value_col]
        for group in groups:
            modes[group[0]] = group[1].mode().iloc[0]

        mask = df[missing_value_col].isna()
        df.loc[mask, missing_value_col] = df.loc[mask, col].map(modes)

    return df

def fill_na_by_median(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    medians: dict[str, int] = {}

    for group in df.groupby('TR.GICSSectorCode')['Total Share Float']:
        total_shares = group[1].dropna().round()
        medians[group[0]] = int(total_shares.median())

    mask = df['Total Share Float'].isna()
    df.loc[mask, 'Total Share Float'] = df.loc[mask, 'TR.GICSSectorCode'].map(medians)

    return df

static_df = fill_na_by_modes(static_df)
static_df = fill_na_by_median(static_df)
static_df['TR.HeadquartersCity'] = static_df['TR.HeadquartersCity'].fillna(HQ)

# Filter and convert historic dataframe

In [42]:
# Detect columns with only boolean-like strings and convert dtype to boolean
def is_bool_string_col(series):
    bool_strings = [{'true', 'false'}, {'True', 'False'}, {'yes', 'no'}, {'Yes', 'No'}, {'1', '0'}]
    values = set(series.dropna().unique())
    return any(values <= s for s in bool_strings)

def transform_bool_strings(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    for col in df.select_dtypes(include=['object', 'string']):
        if is_bool_string_col(df[col]):
            df[col] = df[col].replace(
                {'true': True, 'True': True, 'yes': True, 'Yes': True, '1': True,
                'false': False, 'False': False, 'no': False, 'No': False, '0': False}).astype('boolean')
            df[col] = df[col].astype('boolean')
    return df

historic_df = transform_bool_strings(historic_df)

# remove columns with only one unique (non‑NaN) value
def remove_columns_with_only_one_unique_value(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    nunique = df.nunique(dropna=True)
    cols_to_drop = nunique[nunique == 1].index
    return df.drop(columns=cols_to_drop)

# Merge static and Historic Frame

In [43]:
# Join static and historic dataframe to have GICS Sector Codes in the dataframe
all_df = historic_df.merge(
    static_df,
    on='Instrument',
    how='left'
)

In [ ]:
# OPTIONAL: filter rows with a lot of nan values. 1_788 rows with over 100 nan values
# all_df = all_df.drop(all_df.loc[all_df.isna().sum(axis=1) > 100, all_df.columns].index)

# Compute Median and Mode

In [44]:
# Computing the median not only per sector but also aligning them per year. So every median is computed from values of the same year.
# Only if there is not enough data, then compute only per sector.

import numpy as np

#all_df = all_df.reset_index()

def calculate_median(dataframe: pd.DataFrame) -> pd.DataFrame:
    df: pd.DataFrame = dataframe.copy()
    num_cols = df.select_dtypes(include=['Float64', 'Int64']).columns
    int_cols = df.select_dtypes(include='Int64').columns

    df[int_cols] = df[int_cols].astype('Float64')

    fine_grp = df.groupby(['Date', 'TR.GICSSectorCode'], observed=True)[num_cols]
    coarse_grp = df.groupby('TR.GICSSectorCode', observed=True)[num_cols]

    medians_fine = fine_grp.median().reset_index(drop=True)
    medians_coarse = coarse_grp.median().reset_index(drop=True)

    medians_combined = medians_fine.where(~medians_fine.isna(), medians_coarse)

    mask = df[num_cols].isna()
    df[num_cols] = df[num_cols].where(~mask, medians_combined)
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[int_cols] = df[int_cols].round().astype('Int64')

    return df

def _first_mode(df: pd.DataFrame):
    m = df.mode()
    return m.iloc[0] if not m.empty else np.nan

# Mode for columns with categorical types
def calculate_mode(dataframe: pd.DataFrame) -> pd.DataFrame:
    df: pd.DataFrame = dataframe.copy()
    cat_cols = df.select_dtypes(include=['object', 'category', 'string', 'bool']).columns

    modes_fine = (
        df.groupby(['Date', 'TR.GICSSectorCode'], observed=True)[cat_cols]
        .transform(_first_mode)
    )
    modes_coarse = (
        df.groupby('TR.GICSSectorCode', observed=True)[cat_cols]
        .transform(_first_mode)
    )

    modes_combined = modes_fine.where(~modes_fine.isna(), modes_coarse)
    mask = df[cat_cols].isna()

    df[cat_cols] = df[cat_cols].where(~mask, modes_combined)

    return df

all_df = calculate_median(all_df)
all_df = calculate_mode(all_df)

if all_df.isna().sum().sum() > 0:
    print(f"Mode: {all_df.isna().sum().sum()} is not Zero!!!")

In [45]:
all_df['Instrument'] = all_df['Instrument'].astype('category')
all_df.to_csv(config.data_dir / "datasets" / "median" / "median_imputed.csv")
dtypes: pd.DataFrame = all_df.dtypes.to_frame('dtypes').reset_index()
dtypes.to_csv(config.data_dir / "datasets" / "median" / "median_imputed_dtypes.csv")

boolean     487
Float64     144
category     13
Int64        10
category      2
category      2
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64

# TODO: EMPCA

In [ ]:
from data.cleaning import read_csv
from core import Config
import pandas as pd

config = Config()
all_df = read_csv(
    filepath=config.data_dir / "datasets" / "median" / "median_imputed.csv",
    dtypes_filepath=config.data_dir / "datasets" / "median" / "median_imputed_dtypes.csv",
    index_list=[0]
)

In [ ]:
# one hot encoding
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
x = pd.DataFrame(encoder.fit_transform(all_df.select_dtypes(include=['object', 'category', 'string'])))

In [ ]:
from sklearn.preprocessing import StandardScaler
x = all_df.loc[:, all_df.columns].values
x = StandardScaler().fit_transform(X=x)

In [ ]:
import analysis.empca as e
from wpca import EMPCA
import matplotlib.pyplot as plt

In [ ]:
m = e.empca(data=all_df.select_dtypes(include=['number']))

In [ ]:
empca = EMPCA(n_components=146).fit(X=all_df.select_dtypes(include=['number']))

In [ ]:
plt.plot(empca.components_[:2].T, c='black')

In [ ]:
def plot_results(ThisPCA, X, weights=None, Xtrue=None, ncomp=2):
    # Compute the standard/weighted PCA
    if weights is None:
        kwds = {}
    else:
        kwds = {'weights': weights}

    # Compute the PCA vectors & variance
    pca = ThisPCA(n_components=10).fit(X, **kwds)

    # Reconstruct the data using the PCA model
    Y = ThisPCA(n_components=ncomp).fit_reconstruct(X, **kwds)

    # Create the plots
    fig, ax = plt.subplots(2, 2, figsize=(16, 6))
    if Xtrue is not None:
        ax[0, 0].plot(Xtrue[:10].T, c='gray', lw=1)
        ax[1, 1].plot(Xtrue[:10].T, c='gray', lw=1)
    ax[0, 0].plot(X[:10].T, c='black', lw=1)
    ax[1, 1].plot(Y[:10].T, c='black', lw=1)

    ax[0, 1].plot(pca.components_[:ncomp].T, c='black')

    ax[1, 0].plot(np.arange(1, 11), pca.explained_variance_ratio_)
    ax[1, 0].set_xlim(1, 10)
    ax[1, 0].set_ylim(0, None)

    ax[0, 0].xaxis.set_major_formatter(plt.NullFormatter())
    ax[0, 1].xaxis.set_major_formatter(plt.NullFormatter())

    ax[0, 0].set_title('Input Data')
    ax[0, 1].set_title('First {0} Principal Vectors'.format(ncomp))
    ax[1, 1].set_title('Reconstructed Data ({0} components)'.format(ncomp))
    ax[1, 0].set_title('PCA variance ratio')
    ax[1, 0].set_xlabel('principal vector')
    ax[1, 0].set_ylabel('proportion of total variance')

    fig.suptitle(ThisPCA.__name__, fontsize=16)

plot_results(EMPCA, all_df.select_dtypes(include=['number']))